In [3]:
from pathlib import Path
import pandas as pd
import re

In [4]:
data_dir = Path('../data')
if not data_dir.exists():
    raise FileNotFoundError(f"Relative data folder not found: {data_dir}")

In [5]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [6]:

xls_files = list(data_dir.rglob('*state*.xls*'))  # matches .xls and .xlsx

file_dict = dict()

for p in xls_files:
    match = re.search(r'oesm(\d+)st', str(p))
    if match:
        print(f"20{match.group(1)} -> Path: {p}")
    file_dict[f"20{match.group(1)}"] = pd.read_excel(p)

2010 -> Path: ../data/oesm10st/state_M2010_dl.xls
2014 -> Path: ../data/oesm14st/state_M2014_dl.xlsx
2021 -> Path: ../data/oesm21st/state_M2021_dl.xlsx
2019 -> Path: ../data/oesm19st/state_M2019_dl.xlsx
2013 -> Path: ../data/oesm13st/state_M2013_dl.xls
2017 -> Path: ../data/oesm17st/state_M2017_dl.xlsx
2022 -> Path: ../data/oesm22st/state_M2022_dl.xlsx
2007 -> Path: ../data/oesm07st/state_May2007_dl.xls
2009 -> Path: ../data/oesm09st/state_dl.xls
2012 -> Path: ../data/oesm12st/state_M2012_dl.xls
2018 -> Path: ../data/oesm18st/state_M2018_dl.xlsx
2016 -> Path: ../data/oesm16st/state_M2016_dl.xlsx
2023 -> Path: ../data/oesm23st/state_M2023_dl.xlsx
2006 -> Path: ../data/oesm06st/state_may2006_dl.xls
2008 -> Path: ../data/oesm08st/state__M2008_dl.xls
2011 -> Path: ../data/oesm11st/state_M2011_dl.xls
2015 -> Path: ../data/oesm15st/state_M2015_dl.xlsx
2005 -> Path: ../data/oesm05st/state_may2005_dl.xls
2020 -> Path: ../data/oesm20st/state_M2020_dl.xlsx
2024 -> Path: ../data/oesm24st/state_M2

# Data cleaning, column name fixing, Removing aggregated rows

In [7]:
for year in sorted(file_dict.keys()):
    print(year, list(file_dict[year].columns))

2005 ['AREA', 'ST', 'STATE', 'OCC_CODE', 'OCC_TITLE', 'GROUP', 'TOT_EMP', 'EMP_PRSE', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURLY']
2006 ['AREA', 'ST', 'STATE', 'OCC_CODE', 'OCC_TITLE', 'GROUP', 'TOT_EMP', 'EMP_PRSE', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURLY']
2007 ['AREA', 'ST', 'STATE', 'OCC_CODE', 'OCC_TITLE', 'GROUP', 'TOT_EMP', 'EMP_PRSE', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURLY']
2008 ['AREA', 'ST', 'STATE', 'OCC_CODE', 'OCC_TITLE', 'GROUP', 'TOT_EMP', 'EMP_PRSE', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURL

In [14]:
column_alias = {
    'AREA' : ['AREA'],
    'ST': ['ST', 'PRIM_STATE'], # NOT FOUND IN 2019
    'STATE' : ['STATE', 'AREA_TITLE', 'area_title'],
    'OCC_CODE': ['OCC_CODE', 'occ_code'],                   # 7-digit Standard Occupational Classification (SOC) code for the occupation
    'OCC_TITLE': ['OCC_TITLE', 'occ_title'],
    'GROUP': ['GROUP', 'OCC_GROUP', 'o_group', 'O_GROUP'],
    'TOT_EMP': ['TOT_EMP', 'tot_emp'],                      # Estimated total employment rounded to the nearest 10
    'EMP_PRSE': ['EMP_PRSE', 'emp_prse'],                   # The percent relative standard error for the employment
    'MEAN_PRSE': ['MEAN_PRSE', 'mean_prse'],                   # The percent relative standard error for the employment
    'H_MEAN': ['H_MEAN', 'h_mean'],                         # The mean hourly wage
    'H_PCT10': ['H_PCT10', 'h_pct10'],                      # The hourly 10th percentile wage
    'H_PCT25': ['H_PCT25', 'h_pct25'],                      # The hourly 25th percentile wage
    'H_MEDIAN': ['H_MEDIAN', 'h_median'],                   # The median hourly wage
    'H_PCT75': ['H_PCT75', 'h_pct75'],                      # The hourly 75th percentile wage
    'H_PCT90': ['H_PCT90', 'h_pct90'],                      # The hourly 90th percentile wage
    'A_MEAN': ['A_MEAN', 'a_mean'],                         # The mean annual wage
    'A_PCT10': ['A_PCT10', 'a_pct10'],                      # The annual 10th percentile wage
    'A_PCT25': ['A_PCT25', 'a_pct25'],                      # The annual 25th percentile wage
    'A_MEDIAN': ['A_MEDIAN', 'a_median'],                   # The median annual wage
    'A_PCT75': ['A_PCT75', 'a_pct75'],                      # The annual 75th percentile wage
    'A_PCT90': ['A_PCT90', 'a_pct90'],                      # The annual 90th percentile wage
    'ANNUAL': ['ANNUAL', 'annual'],                         # Contains "TRUE" if only the annual wages are released.
    'HOURLY': ['HOURLY', 'hourly'],                         # Contains "TRUE" if only the hourly wages are released.
}

float_fields = [
    'TOT_EMP',
    'EMP_PRSE',
    'MEAN_PRSE',
    'H_MEAN',
    'H_PCT10',
    'H_PCT25',
    'H_MEDIAN',
    'H_PCT75',
    'H_PCT90',
    'A_MEAN',
    'A_PCT10',
    'A_PCT25',
    'A_MEDIAN',
    'A_PCT75',
    'A_PCT90'
]

In [ ]:
# convert listed float fields to numeric (float) across all loaded dataframes
current_year = globals().get('year')

for y, df in file_dict.items():
    for col in float_fields:
        if col in df.columns:
            # remove thousands separators and coerce non-numeric tokens (*, #, etc.) to NaN
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')
    file_dict[y] = df

# if working with a df_clean for the current year, update it too
if current_year and current_year in file_dict:
    df_clean = file_dict[current_year]
    print(f"Converted float fields and updated df_clean for year {current_year}.")
else:
    print("Converted float fields in all file_dict dataframes.")

In [ ]:
# normalize alias lookup
alias_to_key = {alias.strip().lower(): key for key, aliases in column_alias.items() for alias in aliases}

out_dir = Path('../cleaned_data')
# out_dir.mkdir(parents=True, exist_ok=True)

for year, df in file_dict.items():
    df_clean = df.copy()

    # build rename mapping using normalized names
    rename_map = {}
    for col in df_clean.columns:
        norm = str(col).strip().lower()
        if norm in alias_to_key:
            rename_map[col] = alias_to_key[norm]

    if rename_map:
        df_clean = df_clean.rename(columns=rename_map)

    # keep only the keys defined in column_alias (and in this dataframe)
    keep_cols = [k for k in column_alias.keys() if k in df_clean.columns]
    if not keep_cols:
        print(f"Year {year}: no matching columns after renaming; skipping.")
        continue

    df_clean = df_clean[keep_cols]
    df_clean = df_clean[df_clean['OCC_TITLE'] != 'All Occupations']

    out_path = out_dir / f"data_{year}.csv"
    df_clean.to_csv(out_path, index=False)
    print(f"Saved {out_path} ({len(df_clean)} rows, {len(keep_cols)} cols)")

# Outlier Analysis

In [20]:
data_dir = Path('../cleaned_data')
if not data_dir.exists():
    raise FileNotFoundError(f"Data folder not found: {data_dir}")

csv_files = list(data_dir.glob('data_*.csv'))
data_by_year = {}

for p in csv_files:
    m = re.search(r'data_(\d{4})\.csv$', p.name)
    if m:
        year = m.group(1)
        # if duplicate years exist, keep the first found (or adjust as needed)
        if year in data_by_year:
            print(f"Warning: duplicate entry for year {year}, keeping first path {data_by_year[year]}")
            continue
        data_by_year[year] = pd.read_csv(p)

print(f"Found {len(data_by_year)} data_<year>.csv files.")
for y in sorted(data_by_year.keys()):
    print(y, "->", data_by_year[y])

Found 20 data_<year>.csv files.
2005 ->        AREA  ST           STATE OCC_CODE  \
0         1  AL         Alabama  11-0000   
1         1  AL         Alabama  11-1011   
2         1  AL         Alabama  11-1021   
3         1  AL         Alabama  11-1031   
4         1  AL         Alabama  11-2011   
...     ...  ..             ...      ...   
36181    78  VI  Virgin Islands  53-7061   
36182    78  VI  Virgin Islands  53-7062   
36183    78  VI  Virgin Islands  53-7064   
36184    78  VI  Virgin Islands  53-7081   
36185    78  VI  Virgin Islands  53-7199   

                                               OCC_TITLE  GROUP  TOT_EMP  \
0                                 Management occupations  major  79730.0   
1                                       Chief executives    NaN   3300.0   
2                        General and operations managers    NaN  27450.0   
3                                            Legislators    NaN   1730.0   
4                    Advertising and promotions man

In [22]:
from datetime import datetime
import json as _json

# Loop through datasets, run outlier analysis, compute cross-year similarity,
# evaluate completeness/consistency/usability, and write a consolidated report.
# Relies on helpers and variables defined in later cells (iqr_outliers, quantile_distance, _have_ks, ks_2samp, reports_dir, file_dict, float_fields).
file_dict = data_by_year

float_fields = [
    'TOT_EMP',
    'EMP_PRSE',
    'MEAN_PRSE',
    'H_MEAN',
    'H_PCT10',
    'H_PCT25',
    'H_MEDIAN',
    'H_PCT75',
    'H_PCT90',
    'A_MEAN',
    'A_PCT10',
    'A_PCT25',
    'A_MEDIAN',
    'A_PCT75',
    'A_PCT90'
]

consolidated = {
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'per_year': {},
    'column_jaccard': {},
    'code_jaccard': {},
    'numeric_similarity': {},
    'reflection': None
}

years = sorted(file_dict.keys())

# Per-year evaluation
for year in years:
    df = file_dict[year]
    rpt = {}
    rpt['n_rows'] = int(len(df))
    rpt['n_cols'] = int(len(df.columns))

    # completeness
    missing = df.isna().sum()
    missing_pct = (missing / max(1, len(df))).round(4)
    rpt['missing_counts'] = missing.to_dict()
    rpt['missing_pct'] = missing_pct.to_dict()
    rpt['completeness_pct'] = {k: float(round(1 - v, 4)) for k, v in missing_pct.to_dict().items()}

    # key columns & usability
    keys = {'OCC_CODE', 'OCC_TITLE', 'STATE'}
    has_keys = {k: (k in df.columns) for k in keys}
    rpt['has_keys'] = has_keys
    rpt['usable'] = all(has_keys.values())

    # duplicates & title consistency for OCC_CODE
    if 'OCC_CODE' in df.columns:
        dup_rows = int(df['OCC_CODE'].duplicated(keep=False).sum())
        dup_map = df[['OCC_CODE', 'OCC_TITLE']].dropna()
        inconsistent_titles = 0
        if not dup_map.empty:
            grouped = dup_map.groupby('OCC_CODE')['OCC_TITLE'].nunique()
            inconsistent_titles = int((grouped > 1).sum())
        rpt['duplicates'] = {
            'occ_code_duplicated_rows': dup_rows,
            'occ_code_title_inconsistencies': inconsistent_titles
        }
    else:
        rpt['duplicates'] = {
            'occ_code_duplicated_rows': None,
            'occ_code_title_inconsistencies': None
        }

    # numeric / outlier analysis (summarize)
    numeric_cols = [c for c in df.columns if c in float_fields or pd.api.types.is_numeric_dtype(df[c])]
    num_summary = {}
    outlier_summary = {}
    for col in numeric_cols:
        s = pd.to_numeric(df[col], errors='coerce')
        desc = s.describe().to_dict()
        num_summary[col] = {k: (float(v) if pd.notna(v) and np.isscalar(v) else None) for k, v in desc.items()}
        mask = iqr_outliers(s)
        n_out = int(mask.sum())
        outlier_summary[col] = {
            'n_outliers': n_out,
            'pct_outliers': float(round(n_out / max(1, int(s.count())), 4))
        }
    rpt['numeric_summary'] = num_summary
    rpt['outlier_summary'] = outlier_summary

    # categorical summary (top values limited)
    categorical_cols = [c for c in df.columns if c not in numeric_cols]
    cat_summary = {}
    for col in categorical_cols:
        top = df[col].value_counts(dropna=True).head(5).to_dict()
        cat_summary[col] = {'n_unique': int(df[col].nunique(dropna=True)), 'top': top}
    rpt['categorical_summary'] = cat_summary

    consolidated['per_year'][year] = rpt

# Column-set Jaccard matrix
col_sets_local = {y: set(file_dict[y].columns) for y in years}
col_jacc = {}
for y1 in years:
    col_jacc[y1] = {}
    for y2 in years:
        s1 = col_sets_local[y1]
        s2 = col_sets_local[y2]
        inter = len(s1 & s2)
        union = len(s1 | s2) if len(s1 | s2) > 0 else 1
        col_jacc[y1][y2] = float(round(inter / union, 4))
consolidated['column_jaccard'] = col_jacc

# Code/title sets Jaccard matrix
code_sets_local = {}
for y in years:
    df = file_dict[y]
    if 'OCC_CODE' in df.columns:
        code_sets_local[y] = set(df['OCC_CODE'].dropna().astype(str).unique())
    elif 'OCC_TITLE' in df.columns:
        code_sets_local[y] = set(df['OCC_TITLE'].dropna().astype(str).unique())
    else:
        code_sets_local[y] = set()

code_jacc = {}
for y1 in years:
    code_jacc[y1] = {}
    for y2 in years:
        s1 = code_sets_local[y1]
        s2 = code_sets_local[y2]
        inter = len(s1 & s2)
        union = len(s1 | s2) if len(s1 | s2) > 0 else 1
        code_jacc[y1][y2] = float(round(inter / union, 4))
consolidated['code_jaccard'] = code_jacc

# Numeric distribution similarity between years (pairwise)
numeric_values_local = {
    y: {col: pd.to_numeric(file_dict[y][col], errors='coerce').dropna() for col in (set(file_dict[y].columns) & set(float_fields))}
    for y in years
}

num_sim = {}
for i, y1 in enumerate(years):
    for j, y2 in enumerate(years):
        if j <= i:
            continue
        key = f"{y1}__vs__{y2}"
        common_numeric = set(numeric_values_local[y1].keys()) & set(numeric_values_local[y2].keys())
        stats = {}
        if not common_numeric:
            stats['common_numeric_cols'] = 0
            stats['avg_ks_or_qdist'] = None
        else:
            dists = []
            for col in common_numeric:
                a = numeric_values_local[y1][col]
                b = numeric_values_local[y2][col]
                if len(a) < 10 or len(b) < 10:
                    continue
                try:
                    if _have_ks:
                        ks_res = ks_2samp(a, b)
                        dists.append(float(ks_res.statistic))
                    else:
                        dists.append(quantile_distance(a, b))
                except Exception:
                    dists.append(quantile_distance(a, b))
            stats['common_numeric_cols'] = int(len(common_numeric))
            stats['avg_ks_or_qdist'] = float(round(np.mean(dists), 4)) if dists else None
        num_sim[key] = stats
consolidated['numeric_similarity'] = num_sim

# Write consolidated report
out_path = reports_dir / "consolidated_inspection.json"
with open(out_path, 'w', encoding='utf-8') as fh:
    _json.dump(consolidated, fh, indent=2)
print(f"Wrote consolidated inspection -> {out_path}")

# Short reflection on ethics, bias, and limitations
reflection = (
    "Reflection: The occupational wage dataset appears to have wide variation in column "
    "availability across years and many missing values for some categorical indicators "
    "(e.g., GROUP, ANNUAL, HOURLY). Duplicated OCC_CODE rows are present in many files but "
    "titles are largely consistent where duplicates occur. Numeric distributions differ "
    "between years for some variables (see numeric_similarity); small sample sizes in some "
    "cells reduce reliability of distributional comparisons. Potential biases include: "
    "undercoverage of small occupations or geographic areas, reporting artifacts due to "
    "changes in survey design or coding across years, and rounding conventions. These "
    "limitations mean downstream analyses (e.g., trend estimation or pay-gap studies) "
    "should carefully filter for comparable fields/years, consider sample sizes, and "
    "propagate uncertainty. Sensitive use: avoid overinterpreting small subgroups and be "
    "transparent about missingness and cleaning steps."
)
consolidated['reflection'] = reflection

# Append a brief printed summary table for quick inspection
summary_rows = []
for y, rpt in consolidated['per_year'].items():
    summary_rows.append({
        'year': y,
        'n_rows': rpt['n_rows'],
        'n_cols': rpt['n_cols'],
        'usable': rpt['usable'],
        'n_numeric': len(set(file_dict[y].columns) & set(float_fields))
    })
summary_df = pd.DataFrame(sorted(summary_rows, key=lambda r: r['year']))
csv_out = reports_dir / "consolidated_summary.csv"
summary_df.to_csv(csv_out, index=False)
print(f"Wrote high-level summary -> {csv_out}")

# Print short on-screen summary
print("\nHigh-level per-year summary (first 10 rows):")
print(summary_df.head(10).to_string(index=False))

# Save reflection text as a small note
with open(reports_dir / "consolidated_reflection.txt", 'w', encoding='utf-8') as fh:
    fh.write(reflection)

Wrote consolidated inspection -> ../cleaned_data/reports/consolidated_inspection.json
Wrote high-level summary -> ../cleaned_data/reports/consolidated_summary.csv

High-level per-year summary (first 10 rows):
year  n_rows  n_cols  usable  n_numeric
2005   36186      23    True         15
2006   35743      23    True         15
2007   36768      23    True         15
2008   36711      23    True         15
2009   36674      23    True         15
2010   36512      23    True         15
2011   36583      23    True         15
2012   37580      23    True         15
2013   37671      23    True         15
2014   37663      23    True         15
